# DAY 4 — Fund Performance Analytics
## Capstone Project - I | Bluestock Mutual Fund Analytics

**Covers:**
1. Daily Returns — `daily_return = nav_t / nav_t-1 - 1`
2. CAGR (1yr, 3yr, 5yr) — `CAGR = (NAV_end / NAV_start)^(1/n) - 1`
3. Sharpe Ratio — `(Rp - Rf) / Std(Rp) × √252`, Rf = 6.5%
4. Sortino Ratio — same with downside std deviation
5. Alpha & Beta — OLS regression on Nifty 100 via `scipy.stats.linregress`
6. Maximum Drawdown — `min(NAV / running_max - 1)` with worst date range
7. Fund Scorecard (0–100) composite ranking
8. Benchmark comparison chart + tracking error

**Deliverables:** `fund_scorecard.csv`, `alpha_beta.csv`, benchmark chart PNG

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# Ensure we're in the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Import the Day 4 module
sys.path.insert(0, '.')
import performance_analytics as pa

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Load Data

In [ ]:
fund_master = pa._load('01_fund_master_clean.csv', '01_fund_master.csv')
nav_pivot   = pa.load_nav_pivot()
bench_pivot = pa.load_benchmark_pivot()
print(f'Funds: {nav_pivot.shape[1]}  |  Dates: {nav_pivot.shape[0]}')
nav_pivot.head(3)

## 2. Daily Returns  —  `nav_t / nav_t-1 - 1`

In [ ]:
returns = pa.compute_daily_returns(nav_pivot)

# Distribution plot for first 4 funds
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
name_map = fund_master.set_index('amfi_code')['scheme_name'].str[:30].to_dict()
for ax, code in zip(axes.flat, returns.columns[:4]):
    r = returns[code].dropna()
    ax.hist(r*100, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(name_map.get(code, str(code)), fontsize=9)
    ax.set_xlabel('Daily Return (%)')
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
plt.suptitle('Daily Return Distribution — Sample Funds', fontweight='bold')
plt.tight_layout()
plt.show()
print('Distribution looks reasonable: mean ≈ 0, bell-shaped, no extreme tails')

## 3. CAGR — 1yr, 3yr, 5yr Comparison Table

In [ ]:
cagr_df = pa.compute_cagr(nav_pivot)
cagr_display = cagr_df.merge(
    fund_master[['amfi_code','scheme_name','fund_house','category']], on='amfi_code'
)[['scheme_name','fund_house','category','cagr_1yr_pct','cagr_3yr_pct','cagr_5yr_pct']]
cagr_display = cagr_display.sort_values('cagr_3yr_pct', ascending=False)
print('CAGR Comparison Table (sorted by 3yr CAGR):')
cagr_display.style.format({'cagr_1yr_pct':'{:.2f}%','cagr_3yr_pct':'{:.2f}%','cagr_5yr_pct':'{:.2f}%'})

## 4. Sharpe & Sortino Ratios  (Rf = 6.5%)

In [ ]:
sharpe_df = pa.compute_sharpe_sortino(returns)
sharpe_display = sharpe_df.merge(
    fund_master[['amfi_code','scheme_name']], on='amfi_code'
)[['scheme_name','sharpe_ratio_calc','sortino_ratio_calc']]
sharpe_display = sharpe_display.sort_values('sharpe_ratio_calc', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top = sharpe_display.head(10)
axes[0].barh(top['scheme_name'].str[:25], top['sharpe_ratio_calc'], color='steelblue')
axes[0].set_title('Top 10 by Sharpe Ratio', fontweight='bold')
axes[0].set_xlabel('Sharpe Ratio')
axes[1].barh(top['scheme_name'].str[:25], top['sortino_ratio_calc'], color='seagreen')
axes[1].set_title('Top 10 by Sortino Ratio', fontweight='bold')
axes[1].set_xlabel('Sortino Ratio')
plt.tight_layout()
plt.show()

## 5. Alpha & Beta — OLS Regression on Nifty 100

In [ ]:
alpha_df = pa.compute_alpha_beta(returns, bench_pivot)
ab_display = alpha_df.merge(
    fund_master[['amfi_code','scheme_name','category']], on='amfi_code'
)[['scheme_name','category','alpha_calc','beta_calc','r_squared']]
ab_display = ab_display.sort_values('alpha_calc', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ab_display['alpha_calc'].apply(lambda x: 'steelblue' if x >= 0 else 'crimson')
ax.barh(ab_display['scheme_name'].str[:30], ab_display['alpha_calc'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Annualised Alpha vs Nifty 100 — All 40 Funds', fontweight='bold')
ax.set_xlabel('Alpha (%)')
plt.tight_layout()
plt.show()
print(ab_display.to_string(index=False))

## 6. Maximum Drawdown

In [ ]:
dd_df = pa.compute_max_drawdown(nav_pivot)
dd_display = dd_df.merge(
    fund_master[['amfi_code','scheme_name']], on='amfi_code'
)[['scheme_name','max_drawdown_pct_calc','peak_date','trough_date','drawdown_days']]
dd_display = dd_display.sort_values('max_drawdown_pct_calc')

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(dd_display['scheme_name'].str[:30], dd_display['max_drawdown_pct_calc'],
        color='crimson', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Maximum Drawdown — All 40 Funds', fontweight='bold')
ax.set_xlabel('Max Drawdown (%)')
plt.tight_layout()
plt.show()

## 7. Fund Scorecard (0–100)
Composite = **30%** × 3yr CAGR rank + **25%** × Sharpe rank + **20%** × Alpha rank + **15%** × Expense ratio rank (inverse) + **10%** × Max DD rank (inverse)

In [ ]:
scorecard_df = pa.compute_scorecard(cagr_df, sharpe_df, alpha_df, dd_df, fund_master)
scorecard_df = scorecard_df.merge(sharpe_df[['amfi_code','sortino_ratio_calc']], on='amfi_code', how='left')
scorecard_df = scorecard_df.merge(cagr_df[['amfi_code','cagr_1yr_pct','cagr_5yr_pct']], on='amfi_code', how='left')

fig, ax = plt.subplots(figsize=(12, 10))
colors = plt.cm.RdYlGn(scorecard_df['scorecard_100'] / 100)
bars = ax.barh(scorecard_df['scheme_name'].str[:32], scorecard_df['scorecard_100'],
               color=colors, edgecolor='white')
for bar, score in zip(bars, scorecard_df['scorecard_100']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=8)
ax.set_title('Fund Scorecard (0–100) — All 40 Funds', fontweight='bold', fontsize=13)
ax.set_xlabel('Composite Score')
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

disp_cols = ['scorecard_rank','scheme_name','scorecard_100','cagr_3yr_pct',
             'sharpe_ratio_calc','alpha_calc','max_drawdown_pct_calc']
scorecard_df[disp_cols].head(10)

## 8. Benchmark Comparison Chart + Tracking Error

In [ ]:
pa.chart_benchmark_comparison(nav_pivot, returns, bench_pivot, scorecard_df, fund_master)
from IPython.display import Image
Image('reports/charts/16_benchmark_comparison.png')

## 9. Save Deliverable CSVs

In [ ]:
pa.save_deliverables(scorecard_df, alpha_df, cagr_df, sharpe_df, dd_df, fund_master)
print('\nfund_scorecard.csv preview:')
pd.read_csv('reports/fund_scorecard.csv').head()

## 10. Key Performance Findings

In [ ]:
top1 = scorecard_df.iloc[0]
best_cagr = scorecard_df.nlargest(1,'cagr_3yr_pct').iloc[0]
best_sharpe = scorecard_df.nlargest(1,'sharpe_ratio_calc').iloc[0]
worst_dd = scorecard_df.nsmallest(1,'max_drawdown_pct_calc').iloc[0]

findings = [
    f"🏆 Best overall scorecard: {top1['scheme_name'][:40]} — Score {top1['scorecard_100']:.1f}/100",
    f"📈 Best 3yr CAGR: {best_cagr['scheme_name'][:40]} — {best_cagr['cagr_3yr_pct']:.2f}%",
    f"⚖️  Best risk-adjusted (Sharpe): {best_sharpe['scheme_name'][:40]} — {best_sharpe['sharpe_ratio_calc']:.3f}",
    f"📉 Worst drawdown: {worst_dd['scheme_name'][:40]} — {worst_dd['max_drawdown_pct_calc']:.2f}%",
    f"🔵 Average 3yr CAGR (equity): {scorecard_df[scorecard_df['category']=='Equity']['cagr_3yr_pct'].mean():.2f}%",
    f"🔵 Average 3yr CAGR (debt):   {scorecard_df[scorecard_df['category']=='Debt']['cagr_3yr_pct'].mean():.2f}%",
]
for f in findings:
    print(f)
print('\n✅ Day 4 complete — all deliverables saved in reports/')